In [ ]:
# =========================
# Colab: Install
# =========================
!pip -q install -U transformers accelerate scikit-learn openpyxl

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Any, Dict
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer, set_seed
from sklearn.metrics import accuracy_score, f1_score, classification_report
import transformers

# -------------------------
# 0) CONFIG
# -------------------------
set_seed(42)
print("Transformers version:", transformers.__version__)

MODEL_NAME = "xlm-roberta-base"

TRAIN_PATH = "train.xlsx"
VALID_PATH = "valid.xlsx"
TEST_PATH  = "test.xlsx"

TEXT_COL = "clean_text"
LABEL_DIALECT_COL = "dialect"   # 1=Saudi, 0=Egyptian
LABEL_HATE_COL    = "hate"      # 1=Hate, 0=Not hate
LABEL_TOPIC_COL   = "topic"     # 1=Religious, 0=Political

MAX_LENGTH = 128
BATCH_SIZE = 16
LR = 2e-5
EPOCHS = 3
OUTPUT_DIR = "./xlmr_multilabel_with_emoji_feature"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

id2dialect = {1: "Saudi", 0: "Egyptian"}
id2hate    = {1: "Hate", 0: "Not Hate"}
id2topic   = {1: "Religious", 0: "Political"}

# -------------------------
# 0.1) Emoji helper
# -------------------------
HATE_EMOJIS = set([
    '💦', '🐖', '🐷', '🐽', '👞', '🐕', '🐶', '💩', '🐄', '🐮',
    '🐑', '🐏', '👎', '😡', '🤬', '👺', '👿', '😠'
])

def emoji_flag(text: str) -> int:
    return 1 if any(e in text for e in HATE_EMOJIS) else 0

# -------------------------
# 1) LOAD XLSX + VALIDATE
# -------------------------
def load_excel(path: str) -> pd.DataFrame:
    return pd.read_excel(path, engine="openpyxl")

def load_and_validate_xlsx(path: str) -> pd.DataFrame:
    df = load_excel(path)

    required = {TEXT_COL, LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {missing}\nAvailable columns: {list(df.columns)}")

    df = df.copy()
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()

    for c in [LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL]:
        if df[c].isna().any():
            bad_rows = df[df[c].isna()].head(5)
            raise ValueError(f"{path}: column '{c}' has NaN values. Examples:\n{bad_rows[[TEXT_COL, c]]}")

        df[c] = pd.to_numeric(df[c], errors="raise").astype(int)
        bad = ~df[c].isin([0, 1])
        if bad.any():
            examples = df.loc[bad, [TEXT_COL, c]].head(5)
            raise ValueError(f"{path}: column '{c}' must be 0/1 only. Examples:\n{examples}")

    return df

train_df = load_and_validate_xlsx(TRAIN_PATH)
valid_df = load_and_validate_xlsx(VALID_PATH)
test_df  = load_and_validate_xlsx(TEST_PATH)

print("Rows:", len(train_df), len(valid_df), len(test_df))
print("Train label counts:")
print("dialect:", train_df[LABEL_DIALECT_COL].value_counts().to_dict())
print("hate:",    train_df[LABEL_HATE_COL].value_counts().to_dict())
print("topic:",   train_df[LABEL_TOPIC_COL].value_counts().to_dict())

# -------------------------
# 2) TOKENIZER + DATASET
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiLabelDataset(Dataset):
    """
    labels = [dialect, hate, topic] as float32 vector shape (3,)
    emoji_flag = float32 scalar (0/1)
    """
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        text = row[TEXT_COL]

        enc = self.tok(
            text,
            truncation=True,
            padding=False,
            max_length=self.max_len
        )

        labels = np.array([
            int(row[LABEL_DIALECT_COL]),
            int(row[LABEL_HATE_COL]),
            int(row[LABEL_TOPIC_COL]),
        ], dtype=np.float32)

        item = {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": labels,
            "emoji_flag": float(emoji_flag(text)),
        }

        return item

@dataclass
class DataCollatorMultiLabel:
    tokenizer: Any

    def __call__(self, features):
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.float32)
        eflag  = torch.tensor([f["emoji_flag"] for f in features], dtype=torch.float32)

        for f in features:
            f.pop("labels")
            f.pop("emoji_flag")

        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        batch["labels"] = labels
        batch["emoji_flag"] = eflag
        return batch

train_ds = MultiLabelDataset(train_df, tokenizer, MAX_LENGTH)
valid_ds = MultiLabelDataset(valid_df, tokenizer, MAX_LENGTH)
test_ds  = MultiLabelDataset(test_df, tokenizer, MAX_LENGTH)
collator = DataCollatorMultiLabel(tokenizer)

# -------------------------
# 3) MODEL: Multi-Label XLM-R + emoji feature
# -------------------------
class MultiLabelXLMR(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden + 1, 3)   # [dialect, hate, topic]
        self.loss_fn = nn.BCEWithLogitsLoss()

        # learnable gentle bias for hate logit only
        self.emoji_boost = nn.Parameter(torch.tensor(0.75, dtype=torch.float32))

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        emoji_flag=None
    ):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        pooled = out.last_hidden_state[:, 0, :]
        pooled = self.drop(pooled)

        if emoji_flag is None:
            emoji_flag = torch.zeros(pooled.size(0), device=pooled.device, dtype=pooled.dtype)
        else:
            emoji_flag = emoji_flag.to(pooled.dtype)

        x = torch.cat([pooled, emoji_flag.unsqueeze(1)], dim=1)
        logits = self.classifier(x)  # (B,3)

        logits[:, 1] = logits[:, 1] + self.emoji_boost * emoji_flag

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

model = MultiLabelXLMR(MODEL_NAME).to(device)

# -------------------------
# 4) METRICS
# -------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.asarray(logits)
    labels = np.asarray(labels)

    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    metrics = {}

    # Overall exact-match accuracy across all labels
    metrics["overall_acc"] = np.mean(np.all(labels == preds, axis=1))

    metrics["dialect_acc"] = accuracy_score(labels[:, 0], preds[:, 0])
    metrics["dialect_f1"] = f1_score(labels[:, 0], preds[:, 0], average="macro", zero_division=0)
    metrics["dialect_f1_micro"] = f1_score(labels[:, 0], preds[:, 0], average="micro", zero_division=0)

    metrics["hate_acc"] = accuracy_score(labels[:, 1], preds[:, 1])
    metrics["hate_f1"] = f1_score(labels[:, 1], preds[:, 1], average="macro", zero_division=0)
    metrics["hate_f1_micro"] = f1_score(labels[:, 1], preds[:, 1], average="micro", zero_division=0)

    metrics["topic_acc"] = accuracy_score(labels[:, 2], preds[:, 2])
    metrics["topic_f1"] = f1_score(labels[:, 2], preds[:, 2], average="macro", zero_division=0)
    metrics["topic_f1_micro"] = f1_score(labels[:, 2], preds[:, 2], average="micro", zero_division=0)

    metrics["avg_f1"] = (
        metrics["dialect_f1"] +
        metrics["hate_f1"] +
        metrics["topic_f1"]
    ) / 3.0

    metrics["avg_f1_micro"] = (
        metrics["dialect_f1_micro"] +
        metrics["hate_f1_micro"] +
        metrics["topic_f1_micro"]
    ) / 3.0

    return metrics

# -------------------------
# 5) TrainingArguments
# -------------------------
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

# -------------------------
# 6) TRAIN
# -------------------------
trainer.train()

# -------------------------
# 7) TEST PREDICTIONS
# -------------------------
pred_out = trainer.predict(test_ds)
logits = np.asarray(pred_out.predictions)
labels = np.asarray(pred_out.label_ids)

probs = 1.0 / (1.0 + np.exp(-logits))
preds = (probs >= 0.5).astype(int)

overall_acc = np.mean(np.all(labels == preds, axis=1))
print(f"\nOverall exact-match accuracy (all labels together): {overall_acc:.4f}")

# -------------------------
# 8) PRINT 3 REPORTS
# -------------------------
def print_report_with_micro(y_true, y_pred, title: str):
    rep = classification_report(
        y_true, y_pred,
        digits=4,
        zero_division=0,
        output_dict=True
    )

    f1_micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
    acc = accuracy_score(y_true, y_pred)

    def row(k):
        return rep.get(k, {"precision": 0.0, "recall": 0.0, "f1-score": 0.0, "support": 0})

    r0 = row("0")
    r1 = row("1")
    rmacro = row("macro avg")
    rweight = row("weighted avg")
    total_support = int(r0["support"]) + int(r1["support"])

    print(f"\n--- {title} ---")
    print(f"{'':>14}{'precision':>10}{'recall':>10}{'f1-score':>10}{'f1-micro':>10}{'support':>10}")
    print(f"{'0.0':>14}{r0['precision']:>10.4f}{r0['recall']:>10.4f}{r0['f1-score']:>10.4f}{f1_micro:>10.4f}{int(r0['support']):>10}")
    print(f"{'1.0':>14}{r1['precision']:>10.4f}{r1['recall']:>10.4f}{r1['f1-score']:>10.4f}{f1_micro:>10.4f}{int(r1['support']):>10}")
    print()
    print(f"{'accuracy':>14}{'':>10}{'':>10}{acc:>10.4f}{f1_micro:>10.4f}{total_support:>10}")
    print(f"{'macro avg':>14}{rmacro['precision']:>10.4f}{rmacro['recall']:>10.4f}{rmacro['f1-score']:>10.4f}{f1_micro:>10.4f}{int(rmacro['support']):>10}")
    print(f"{'weighted avg':>14}{rweight['precision']:>10.4f}{rweight['recall']:>10.4f}{rweight['f1-score']:>10.4f}{f1_micro:>10.4f}{int(rweight['support']):>10}")

print_report_with_micro(labels[:, 0].astype(int), preds[:, 0].astype(int), "Dialect report (1=saudi,0=egyptian)")
print_report_with_micro(labels[:, 1].astype(int), preds[:, 1].astype(int), "Hate report (1=hate,0=not hate)")
print_report_with_micro(labels[:, 2].astype(int), preds[:, 2].astype(int), "Topic report (1=religious,0=political)")

# -------------------------
# 9) SAVE
# -------------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)
tokenizer.save_pretrained(OUTPUT_DIR)
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "pytorch_model.bin"))
print("\nSaved to:", OUTPUT_DIR)
print("Learned emoji_boost (final):", float(model.emoji_boost.detach().cpu().item()))

# -------------------------
# 10) INTERACTIVE MODE
# -------------------------
def predict(text: str, threshold: float = 0.5):
    model.eval()

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(device)

    eflag = torch.tensor([float(emoji_flag(text))], device=device)

    with torch.no_grad():
        out = model(**enc, emoji_flag=eflag)

    probs = torch.sigmoid(out["logits"]).squeeze(0)
    pred = (probs >= threshold).long()

    dialect_id = int(pred[0].item())
    hate_id    = int(pred[1].item())
    topic_id   = int(pred[2].item())

    dialect_p = float(probs[0].item())
    hate_p    = float(probs[1].item())
    topic_p   = float(probs[2].item())

    return {
        "dialect_id": dialect_id,
        "hate_id": hate_id,
        "topic_id": topic_id,

        "dialect_label": id2dialect[dialect_id],
        "hate_label": id2hate[hate_id],
        "topic_label": id2topic[topic_id],

        "dialect_probs": {
            "Egyptian": 1 - dialect_p,
            "Saudi": dialect_p,
        },
        "hate_probs": {
            "Not Hate": 1 - hate_p,
            "Hate": hate_p,
        },
        "topic_probs": {
            "Political": 1 - topic_p,
            "Religious": topic_p,
        },

        "dialect_confidence": dialect_p if dialect_id == 1 else (1 - dialect_p),
        "hate_confidence": hate_p if hate_id == 1 else (1 - hate_p),
        "topic_confidence": topic_p if topic_id == 1 else (1 - topic_p),

        "emoji_flag": int(emoji_flag(text)),
    }

print("\n=== Interactive Mode ===")
print("Type: exit / quit / stop to end\n")

while True:
    text = input("Tweet> ").strip()
    if text.lower() in {"exit", "quit", "stop"}:
        print("Done.")
        break
    if not text:
        continue

    result = predict(text, threshold=0.5)

    print(
        f"Dialect: {result['dialect_label']} "
        f"(confidence={result['dialect_confidence']:.3f}, "
        f"Egyptian={result['dialect_probs']['Egyptian']:.3f}, "
        f"Saudi={result['dialect_probs']['Saudi']:.3f})"
    )

    print(
        f"Hate: {result['hate_label']} "
        f"(confidence={result['hate_confidence']:.3f}, "
        f"Not Hate={result['hate_probs']['Not Hate']:.3f}, "
        f"Hate={result['hate_probs']['Hate']:.3f}, "
        f"emoji_flag={result['emoji_flag']})"
    )

    print(
        f"Topic: {result['topic_label']} "
        f"(confidence={result['topic_confidence']:.3f}, "
        f"Political={result['topic_probs']['Political']:.3f}, "
        f"Religious={result['topic_probs']['Religious']:.3f})"
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 56.9 MB/s eta 0:00:00
Transformers version: 5.5.4
Device: cuda
Rows: 2773 595 594
Train label counts:
dialect: {1: 1417, 0: 1356}
hate: {1: 1443, 0: 1330}
topic: {0: 1407, 1: 1366}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_1565/1870257419.py:148: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  labels = torch.tensor([f["labels"] for f in features], dtype=torch.float32)


Epoch,Training Loss,Validation Loss,Overall Acc,Dialect Acc,Dialect F1,Dialect F1 Micro,Hate Acc,Hate F1,Hate F1 Micro,Topic Acc,Topic F1,Topic F1 Micro,Avg F1,Avg F1 Micro
1,0.496177,0.387128,0.435294,0.805042,0.802971,0.805042,0.564706,0.501409,0.564706,0.963025,0.963023,0.963025,0.755801,0.777591
2,0.348675,0.333765,0.586555,0.838655,0.837886,0.838655,0.715966,0.707015,0.715966,0.974790,0.974776,0.974790,0.839892,0.843137
3,0.297255,0.303135,0.615126,0.843697,0.843057,0.843697,0.737815,0.737191,0.737815,0.979832,0.979831,0.979832,0.853360,0.853782



Overall exact-match accuracy (all labels together): 0.6128

--- Dialect report (1=saudi,0=egyptian) ---
               precision    recall  f1-score  f1-micro   support
           0.0    0.9264    0.7516    0.8299    0.8350       318
           1.0    0.7649    0.9312    0.8399    0.8350       276

      accuracy                        0.8350    0.8350       594
     macro avg    0.8456    0.8414    0.8349    0.8350       594
  weighted avg    0.8513    0.8350    0.8345    0.8350       594

--- Hate report (1=hate,0=not hate) ---
               precision    recall  f1-score  f1-micro   support
           0.0    0.6845    0.7931    0.7348    0.7205       290
           1.0    0.7674    0.6513    0.7046    0.7205       304

      accuracy                        0.7205    0.7205       594
     macro avg    0.7260    0.7222    0.7197    0.7205       594
  weighted avg    0.7270    0.7205    0.7194    0.7205       594

--- Topic report (1=religious,0=political) ---
               precision